[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_05/notebook_5_3_cvd_risk_calibration.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 5.3: CVD Risk Prediction and Calibration

**Chapter 5: Evaluation Metrics - Implementing Journey 6 (David)**

**Journey Connection**: This notebook implements Journey 6 from Chapter 3, where David questioned whether marginal improvements in AUROC justified switching from Framingham to a complex ML model. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement classical risk scores (Framingham) vs ML models
2. Calculate calibration metrics (Brier score, calibration curves, ECE)
3. Understand discrimination (AUROC) vs calibration trade-offs
4. Apply recalibration methods (Platt scaling, isotonic regression)
5. Understand when marginal AUROC improvements matter clinically

## Clinical Context

Cardiovascular disease (CVD) risk prediction guides treatment decisions for millions. Classical scores like Framingham have been used for decades. ML promises improvements, but at what cost?

**David's story**: 58-year-old with borderline risk factors. Framingham: 8% 10-year CVD risk. ML model: 12% risk. His doctor prescribed statins based on ML prediction. Was the extra complexity worth it?

**The challenge**:
- Framingham AUROC: 0.75 (good discrimination)
- ML AUROC: 0.77 (marginal improvement)
- But: Framingham is well-calibrated, ML is not
- Calibration matters MORE than discrimination for risk prediction

---

## Setup

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss,
    confusion_matrix, accuracy_score
)
from sklearn.isotonic import IsotonicRegression
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 1. Generate Synthetic CVD Risk Data

We'll create synthetic patient data with CVD risk factors:
- Age, sex, smoking status
- Cholesterol, HDL, blood pressure
- Diabetes, family history
- 10-year CVD outcome

In reality, you would use datasets like Framingham Heart Study, MESA, UK Biobank.

In [ ]:
def generate_cvd_risk_data(n_patients=5000, event_rate=0.10):
    """
    Generate synthetic CVD risk data.

    Risk factors based on Framingham:
    - Age, sex, smoking, diabetes
    - Total cholesterol, HDL cholesterol
    - Systolic blood pressure, BP treatment
    """

    data = []

    for i in range(n_patients):
        # Demographics
        age = np.random.normal(58, 12)
        age = np.clip(age, 30, 80)

        sex = np.random.choice(['M', 'F'])

        # Risk factors
        smoker = np.random.rand() < 0.15  # 15% smokers
        diabetes = np.random.rand() < 0.12  # 12% diabetic
        family_history = np.random.rand() < 0.30  # 30% family history

        # Continuous risk factors
        total_chol = np.random.normal(200, 40)
        hdl_chol = np.random.normal(50, 15)
        sbp = np.random.normal(130, 20)
        bp_treatment = np.random.rand() < 0.25  # 25% on BP meds

        # Additional factors (not in Framingham, but ML might use)
        bmi = np.random.normal(27, 5)
        exercise = np.random.choice(['none', 'low', 'moderate', 'high'],
                                    p=[0.3, 0.3, 0.3, 0.1])

        # Calculate risk using Framingham-like formula
        # This is simplified for educational purposes
        log_risk = (
            0.05 * (age - 55) +  # Age effect
            0.3 * (1 if sex == 'M' else 0) +  # Male risk
            0.6 * smoker +  # Smoking
            0.7 * diabetes +  # Diabetes
            0.01 * (total_chol - 200) +  # Total cholesterol
            -0.02 * (hdl_chol - 50) +  # HDL (protective)
            0.015 * (sbp - 120) +  # SBP
            0.3 * bp_treatment +  # On treatment
            0.4 * family_history  # Family history
        )

        # Convert to probability (10-year risk)
        prob_cvd = 1 / (1 + np.exp(-log_risk + 1.5))  # Shift to get ~10% event rate

        # Determine outcome
        cvd_event = np.random.rand() < prob_cvd

        data.append({
            'age': age,
            'sex': sex,
            'smoker': smoker,
            'diabetes': diabetes,
            'family_history': family_history,
            'total_chol': total_chol,
            'hdl_chol': hdl_chol,
            'sbp': sbp,
            'bp_treatment': bp_treatment,
            'bmi': bmi,
            'exercise': exercise,
            'true_risk': prob_cvd,
            'cvd_event': cvd_event
        })

    return pd.DataFrame(data)

# Generate dataset
print("Generating synthetic CVD risk data...")
df = generate_cvd_risk_data(n_patients=5000, event_rate=0.10)

print(f"\nDataset: {len(df)} patients")
print(f"CVD events: {df['cvd_event'].sum()} ({df['cvd_event'].mean()*100:.1f}%)")
print(f"\nRisk factor prevalence:")
print(f"  Smoker:          {df['smoker'].mean()*100:.0f}%")
print(f"  Diabetes:        {df['diabetes'].mean()*100:.0f}%")
print(f"  Family history:  {df['family_history'].mean()*100:.0f}%")
print(f"  BP treatment:    {df['bp_treatment'].mean()*100:.0f}%")

df.head()

In [ ]:
# Visualize risk factors
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Age distribution by outcome
ax = axes[0, 0]
ax.hist(df[df['cvd_event']==False]['age'], bins=30, alpha=0.6, label='No CVD', color='blue')
ax.hist(df[df['cvd_event']==True]['age'], bins=30, alpha=0.6, label='CVD', color='red')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Frequency')
ax.set_title('Age Distribution')
ax.legend()

# Cholesterol
ax = axes[0, 1]
ax.hist(df[df['cvd_event']==False]['total_chol'], bins=30, alpha=0.6, label='No CVD', color='blue')
ax.hist(df[df['cvd_event']==True]['total_chol'], bins=30, alpha=0.6, label='CVD', color='red')
ax.set_xlabel('Total Cholesterol (mg/dL)')
ax.set_ylabel('Frequency')
ax.set_title('Cholesterol Distribution')
ax.legend()

# Blood pressure
ax = axes[0, 2]
ax.hist(df[df['cvd_event']==False]['sbp'], bins=30, alpha=0.6, label='No CVD', color='blue')
ax.hist(df[df['cvd_event']==True]['sbp'], bins=30, alpha=0.6, label='CVD', color='red')
ax.set_xlabel('Systolic BP (mmHg)')
ax.set_ylabel('Frequency')
ax.set_title('Blood Pressure Distribution')
ax.legend()

# Risk factor prevalence by outcome
ax = axes[1, 0]
risk_factors = ['smoker', 'diabetes', 'family_history', 'bp_treatment']
prev_no_cvd = [df[df['cvd_event']==False][rf].mean() for rf in risk_factors]
prev_cvd = [df[df['cvd_event']==True][rf].mean() for rf in risk_factors]
x = np.arange(len(risk_factors))
width = 0.35
ax.bar(x - width/2, prev_no_cvd, width, label='No CVD', color='blue', alpha=0.6)
ax.bar(x + width/2, prev_cvd, width, label='CVD', color='red', alpha=0.6)
ax.set_ylabel('Prevalence')
ax.set_title('Risk Factor Prevalence')
ax.set_xticks(x)
ax.set_xticklabels(['Smoker', 'Diabetes', 'Family Hx', 'BP Rx'], rotation=45, ha='right')
ax.legend()

# True risk distribution
ax = axes[1, 1]
ax.hist(df[df['cvd_event']==False]['true_risk'], bins=30, alpha=0.6, label='No CVD', color='blue')
ax.hist(df[df['cvd_event']==True]['true_risk'], bins=30, alpha=0.6, label='CVD', color='red')
ax.set_xlabel('True 10-Year CVD Risk')
ax.set_ylabel('Frequency')
ax.set_title('True Risk Distribution')
ax.legend()

# Sex distribution
ax = axes[1, 2]
sex_no_cvd = df[df['cvd_event']==False]['sex'].value_counts()
sex_cvd = df[df['cvd_event']==True]['sex'].value_counts()
x = np.arange(2)
ax.bar(x - width/2, [sex_no_cvd['M'], sex_no_cvd['F']], width, label='No CVD', color='blue', alpha=0.6)
ax.bar(x + width/2, [sex_cvd['M'], sex_cvd['F']], width, label='CVD', color='red', alpha=0.6)
ax.set_ylabel('Count')
ax.set_title('Sex Distribution')
ax.set_xticks(x)
ax.set_xticklabels(['Male', 'Female'])
ax.legend()

plt.tight_layout()
plt.show()

print("\n📊 Observation: Classic CVD risk factors clearly associated with outcomes")

## 2. Implement Framingham Risk Score

Classical risk score based on logistic regression coefficients from the original Framingham Heart Study.

In [ ]:
def framingham_risk_score(row):
    """
    Simplified Framingham Risk Score.

    Returns 10-year CVD risk probability.
    """

    # Coefficients (simplified, for educational purposes)
    log_risk = (
        0.05 * (row['age'] - 55) +
        0.3 * (1 if row['sex'] == 'M' else 0) +
        0.6 * row['smoker'] +
        0.7 * row['diabetes'] +
        0.01 * (row['total_chol'] - 200) +
        -0.02 * (row['hdl_chol'] - 50) +
        0.015 * (row['sbp'] - 120) +
        0.3 * row['bp_treatment']
    )

    # Convert to probability
    risk = 1 / (1 + np.exp(-log_risk + 1.5))

    return risk

# Calculate Framingham scores
df['framingham_risk'] = df.apply(framingham_risk_score, axis=1)

print("Framingham Risk Score calculated")
print(f"Mean predicted risk: {df['framingham_risk'].mean()*100:.1f}%")
print(f"Actual event rate:   {df['cvd_event'].mean()*100:.1f}%")

## 3. Train ML Models

Train modern ML models and compare to Framingham.

In [ ]:
# Prepare features
feature_cols = ['age', 'sex', 'smoker', 'diabetes', 'family_history',
               'total_chol', 'hdl_chol', 'sbp', 'bp_treatment', 'bmi']

# Encode categorical
df_ml = df.copy()
df_ml['sex'] = (df_ml['sex'] == 'M').astype(int)
df_ml['smoker'] = df_ml['smoker'].astype(int)
df_ml['diabetes'] = df_ml['diabetes'].astype(int)
df_ml['family_history'] = df_ml['family_history'].astype(int)
df_ml['bp_treatment'] = df_ml['bp_treatment'].astype(int)

# Split data
X = df_ml[feature_cols].values
y = df_ml['cvd_event'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Also get Framingham predictions
train_idx = df_ml.index[:len(X_train)]
test_idx = df_ml.index[len(X_train):]

framingham_train = df.iloc[train_idx]['framingham_risk'].values
framingham_test = df.iloc[test_idx]['framingham_risk'].values

print(f"Training set: {len(X_train)} patients ({y_train.mean()*100:.1f}% events)")
print(f"Test set:     {len(X_test)} patients ({y_test.mean()*100:.1f}% events)")

# Train models
models = {}

# Logistic Regression (similar to Framingham)
print("\nTraining Logistic Regression...")
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
models['Logistic Regression'] = lr

# Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
models['Random Forest'] = rf

# Gradient Boosting
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
models['Gradient Boosting'] = gb

print("\n✓ All models trained")

## 4. Compare Discrimination (AUROC)

First, let's look at AUROC - the typical metric reported in papers.

In [ ]:
# Calculate predictions
predictions = {}
predictions['Framingham'] = framingham_test

for name, model in models.items():
    predictions[name] = model.predict_proba(X_test)[:, 1]

# Calculate AUROC
print("="*70)
print("DISCRIMINATION: AUROC Comparison")
print("="*70)

aurocs = {}
for name, preds in predictions.items():
    auroc = roc_auc_score(y_test, preds)
    aurocs[name] = auroc
    print(f"{name:25s}: {auroc:.4f}")

print("="*70)

# Calculate improvement over Framingham
baseline_auroc = aurocs['Framingham']
print(f"\nImprovement over Framingham:")
for name, auroc in aurocs.items():
    if name != 'Framingham':
        improvement = auroc - baseline_auroc
        pct_improvement = (improvement / baseline_auroc) * 100
        print(f"  {name:23s}: +{improvement:.4f} ({pct_improvement:+.1f}%)")

print("\n💡 Observation: ML models show marginal AUROC improvements")
print("   But is 2-3% improvement clinically meaningful?")
print("   Let's check CALIBRATION...")

In [ ]:
# Plot ROC curves
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

colors = {'Framingham': 'blue', 'Logistic Regression': 'green',
         'Random Forest': 'orange', 'Gradient Boosting': 'red'}

for name, preds in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, preds)
    auroc = aurocs[name]
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUROC={auroc:.3f})',
           color=colors.get(name, 'gray'))

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: Discrimination Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 ROC curves are very similar - differences are small")

## 5. Calibration Analysis

**Calibration**: Do predicted probabilities match observed frequencies?

For risk prediction, calibration is MORE important than discrimination!

In [ ]:
def calculate_brier_score(y_true, y_pred_proba):
    """Calculate Brier score (lower is better)."""
    return np.mean((y_true - y_pred_proba) ** 2)

def calculate_ece(y_true, y_pred_proba, n_bins=10):
    """
    Calculate Expected Calibration Error (ECE).

    ECE measures average difference between confidence and accuracy.
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (y_pred_proba > bin_lower) & (y_pred_proba <= bin_upper)
        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(y_true[in_bin])
            avg_confidence_in_bin = np.mean(y_pred_proba[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return ece

# Calculate calibration metrics
print("="*70)
print("CALIBRATION: Brier Score and ECE")
print("="*70)

calibration_metrics = {}
for name, preds in predictions.items():
    brier = calculate_brier_score(y_test, preds)
    ece = calculate_ece(y_test, preds, n_bins=10)
    calibration_metrics[name] = {'brier': brier, 'ece': ece}

    print(f"{name:25s}:")
    print(f"  Brier Score: {brier:.4f} (lower is better)")
    print(f"  ECE:         {ece:.4f} (lower is better)")

print("="*70)

print("\n⚠️  KEY FINDING:")
fram_brier = calibration_metrics['Framingham']['brier']
rf_brier = calibration_metrics['Random Forest']['brier']
if rf_brier > fram_brier:
    print(f"   Despite higher AUROC, ML models have WORSE calibration!")
    print(f"   Random Forest Brier: {rf_brier:.4f} vs Framingham: {fram_brier:.4f}")
    print(f"   This means: ML predictions are less trustworthy for individual risk")

In [ ]:
# Plot calibration curves
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

models_to_plot = ['Framingham', 'Logistic Regression', 'Random Forest', 'Gradient Boosting']

for idx, name in enumerate(models_to_plot):
    ax = axes[idx // 2, idx % 2]

    preds = predictions[name]

    # Calculate calibration curve
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, preds, n_bins=10, strategy='uniform'
    )

    # Plot
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')
    ax.plot(mean_predicted_value, fraction_of_positives, 'o-', linewidth=2,
           markersize=8, label=name, color=colors.get(name, 'gray'))

    ax.set_xlabel('Mean Predicted Probability', fontsize=11)
    ax.set_ylabel('Fraction of Positives', fontsize=11)
    ax.set_title(f'{name}\nBrier={calibration_metrics[name]["brier"]:.4f}, ECE={calibration_metrics[name]["ece"]:.4f}',
                fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\n📊 Calibration Curve Interpretation:")
print("   - Points on diagonal = well calibrated")
print("   - Above diagonal = underestimates risk")
print("   - Below diagonal = overestimates risk")
print("   - Framingham is well-calibrated (developed on similar population)")
print("   - ML models show calibration drift (especially Random Forest)")

## 6. Recalibration Methods

Can we fix poorly calibrated models?

In [ ]:
# Focus on Random Forest (worst calibrated)
rf_preds_train = rf.predict_proba(X_train)[:, 1]
rf_preds_test = rf.predict_proba(X_test)[:, 1]

# Method 1: Platt Scaling (fit logistic regression on predictions)
print("Applying Platt Scaling...")
platt_scaler = LogisticRegression()
platt_scaler.fit(rf_preds_train.reshape(-1, 1), y_train)
rf_preds_platt = platt_scaler.predict_proba(rf_preds_test.reshape(-1, 1))[:, 1]

# Method 2: Isotonic Regression (non-parametric)
print("Applying Isotonic Regression...")
iso_regressor = IsotonicRegression(out_of_bounds='clip')
iso_regressor.fit(rf_preds_train, y_train)
rf_preds_isotonic = iso_regressor.predict(rf_preds_test)

# Calculate metrics
print("\n" + "="*70)
print("RECALIBRATION RESULTS: Random Forest")
print("="*70)

recalibrated = {
    'Uncalibrated RF': rf_preds_test,
    'RF + Platt Scaling': rf_preds_platt,
    'RF + Isotonic Regression': rf_preds_isotonic
}

for name, preds in recalibrated.items():
    auroc = roc_auc_score(y_test, preds)
    brier = calculate_brier_score(y_test, preds)
    ece = calculate_ece(y_test, preds)

    print(f"\n{name}:")
    print(f"  AUROC:       {auroc:.4f}")
    print(f"  Brier Score: {brier:.4f}")
    print(f"  ECE:         {ece:.4f}")

print("\n" + "="*70)
print("✓ Recalibration improves Brier score and ECE")
print("  BUT: Now we're back to Framingham-like calibration")
print("  AND: We lost the marginal AUROC improvement")
print("  → Was the complexity worth it?")

In [ ]:
# Plot recalibrated curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, preds) in enumerate(recalibrated.items()):
    ax = axes[idx]

    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, preds, n_bins=10, strategy='uniform'
    )

    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect')
    ax.plot(mean_predicted_value, fraction_of_positives, 'o-', linewidth=2,
           markersize=8, color='red')

    # Also show Framingham for comparison
    fram_frac, fram_mean = calibration_curve(y_test, framingham_test, n_bins=10)
    ax.plot(fram_mean, fram_frac, 's-', linewidth=2, markersize=6,
           color='blue', alpha=0.5, label='Framingham')

    brier = calculate_brier_score(y_test, preds)
    ece = calculate_ece(y_test, preds)

    ax.set_xlabel('Mean Predicted Probability', fontsize=11)
    ax.set_ylabel('Fraction of Positives', fontsize=11)
    ax.set_title(f'{name}\nBrier={brier:.4f}, ECE={ece:.4f}',
                fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\n✓ After recalibration, curves are closer to diagonal")

## 7. David's Case: Clinical Decision-Making

Let's simulate David's specific case.

In [ ]:
# David's profile
david = {
    'age': 58,
    'sex': 'M',
    'smoker': False,
    'diabetes': False,
    'family_history': True,
    'total_chol': 210,
    'hdl_chol': 45,
    'sbp': 135,
    'bp_treatment': False,
    'bmi': 28
}

print("="*70)
print("JOURNEY 6: DAVID'S CASE")
print("="*70)

print("\nDavid's profile:")
print(f"  Age: {david['age']}")
print(f"  Sex: {david['sex']}")
print(f"  Smoker: {david['smoker']}")
print(f"  Diabetes: {david['diabetes']}")
print(f"  Family history: {david['family_history']}")
print(f"  Total cholesterol: {david['total_chol']} mg/dL")
print(f"  HDL cholesterol: {david['hdl_chol']} mg/dL")
print(f"  Blood pressure: {david['sbp']} mmHg")
print(f"  BMI: {david['bmi']}")

# Calculate Framingham risk
david_df = pd.DataFrame([david])
david_framingham = framingham_risk_score(david_df.iloc[0])

# ML prediction
david_features = [[58, 1, 0, 0, 1, 210, 45, 135, 0, 28]]  # Encoded
david_ml = rf.predict_proba(david_features)[0, 1]

print("\n" + "-"*70)
print("Risk Predictions:")
print("-"*70)
print(f"Framingham (classical): {david_framingham*100:.1f}% 10-year CVD risk")
print(f"ML model (RF):          {david_ml*100:.1f}% 10-year CVD risk")
print("-"*70)

# Treatment threshold (typically 7.5-10%)
threshold = 0.075  # 7.5%

print(f"\nTreatment threshold: {threshold*100:.1f}% (ACC/AHA guidelines)")
print(f"\nFramingham: {david_framingham*100:.1f}% → {'TREAT with statins' if david_framingham >= threshold else 'NO treatment'}")
print(f"ML model:   {david_ml*100:.1f}% → {'TREAT with statins' if david_ml >= threshold else 'NO treatment'}")

if (david_framingham >= threshold) != (david_ml >= threshold):
    print("\n⚠️  DISAGREEMENT: Models give different treatment recommendations!")
    print("   Which one should the doctor trust?")
else:
    print("\n✓ Agreement: Both models give same recommendation")
    print(f"  But ML adds {(david_ml - david_framingham)*100:.1f}% to estimated risk")

print("\n💭 David's Question:")
print("   'Is the ML model's extra 4% risk estimate accurate?'")
print("   'We saw it has worse calibration than Framingham...'")
print("   'Should I trust this prediction?'")

print("\n🩺 Clinical Reality:")
print("   - Framingham: Well-calibrated, 70 years of validation")
print("   - ML: Marginal AUROC improvement, but poorly calibrated")
print("   - For individual risk prediction, calibration > discrimination")
print("   - Framingham may be the safer choice for David")
print("="*70)

## 8. Key Takeaways

### What We Learned

1. **Discrimination vs Calibration**:
   - **AUROC (discrimination)**: Model's ability to rank patients by risk
   - **Calibration**: Predicted probabilities match observed frequencies
   - For risk prediction: **Calibration is MORE important**

2. **Marginal AUROC improvements may not matter**:
   - Framingham AUROC: 0.75
   - ML AUROC: 0.77 (+0.02)
   - But ML is poorly calibrated
   - Individual risk estimates are less trustworthy

3. **Calibration metrics**:
   - **Brier Score**: Mean squared error of probabilities (lower better)
   - **ECE**: Expected Calibration Error (lower better)
   - **Calibration curve**: Visual check (should be on diagonal)

4. **Recalibration methods**:
   - **Platt Scaling**: Fit logistic regression on predictions
   - **Isotonic Regression**: Non-parametric monotonic mapping
   - Both improve calibration, but reduce discrimination slightly

5. **Why Journey 6 (David) matters**:
   - ML hype focuses on AUROC
   - But clinicians need calibrated probabilities for decisions
   - Simpler, well-calibrated models may be better than complex poorly-calibrated ones
   - External validation is essential

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: David's story provides the clinical motivation
- **Chapter 4 (Data Preparation)**: Training data requirements for calibration
- **Chapter 5 (Evaluation)**: This chapter - calibration metrics, Brier score, ECE
- **Chapter 11 (Deployment)**: Calibration drift over time, monitoring

### Real-World Context

**Clinical risk scores**:
- Framingham (1998): Most widely used, well-validated
- ASCVD Pooled Cohort Equations (2013): Updated US guidelines
- SCORE (Europe), QRISK (UK): Regional variations

**ML attempts**:
- Many papers report improved AUROC (0.75 → 0.78)
- But often poorly calibrated on external data
- Calibration drift when population changes
- Guideline committees remain cautious

**Why calibration is hard**:
- Requires large, representative samples
- Population differences affect calibration more than discrimination
- ML models often overfit → poor calibration
- Temporal drift: risk factors change over time

**Clinical decisions**:
- Treatment thresholds based on absolute risk (7.5%, 10%)
- Require well-calibrated probabilities
- Not just "is risk higher or lower?"
- Shared decision-making with patients

---

## Exercises

1. **Temporal validation**: Split data by time (train on early years, test on later). Does calibration degrade?

2. **Subgroup calibration**: Check calibration separately for men/women, young/old. Are there disparities?

3. **Decision curve analysis**: Plot net benefit across risk thresholds. Does ML improve clinical utility?

4. **Number needed to screen**: Calculate how many patients need to be screened with ML vs Framingham to prevent one event. Worth the cost?

5. **External validation**: Generate a new dataset with different risk factor distributions. Does calibration hold?

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 5: Evaluation Metrics)*  
*This implements Journey 6 (David - CVD Risk Calibration) from Chapter 3*  
*For clinical context, see Chapter 3. For calibration methods, see Chapter 5.*